In [0]:
import json
import mlflow
import pandas as pd

with open("../eval/eval_set.json") as f:
    eval_data = pd.DataFrame(json.load(f))

model_uri = "models:/fda_rag.gold.fda_assistant/1"

with mlflow.start_run(run_name="eval-v1-cited"):
    results = mlflow.evaluate(
        model=model_uri,
        data=eval_data,
        model_type="question-answering",
        predictions="answer",
        extra_metrics=[
            mlflow.metrics.genai.faithfulness(model="endpoints:/databricks-gpt-5-5-pro"),
            mlflow.metrics.genai.relevance(model="endpoints:/databricks-gpt-5-5-pro"),
            mlflow.metrics.latency(),
        ])

    print("Aggregate metrics:")
    for k, v in results.metrics.items():
        print(f"  {k}: {v}")

    # Per-row results table — screenshot this for your case study
    display(results.tables["eval_results_table"])